# T2 + T3: Kaggle smoke test and T4 measurements (M1)

Run this **once**, on account A, before launching any real run.

**Kaggle settings:** Accelerator **GPU T4 x2**, Internet **on**, Persistence off.
**Input to attach:** the Tiny-ImageNet dataset `akash2sharma/tiny-imagenet` (Add Input -> Datasets).
CIFAR-10 is downloaded automatically.

What it checks (each section prints PASS/FAIL):
1. Unit tests pass on Kaggle (`pytest`).
2. Every model/config in the run plan trains for 1 epoch on the **real** datasets on a T4 (5 train /
   2 val / 2 test batches), writes all artifacts, and KD students find their same-seed teacher.
3. The aggregation script reads the smoke runs and runs a paired comparison.
4. Whether two training jobs on the two T4s at once are worth it (per-job slowdown).
5. M1: T4 latency (eager + CUDA Graph), per-batch-size peak memory and MACs for the 7 CIFAR models,
   with self-checks against the first-submission measurements.

Run with **Save Version -> Save & Run All (Commit)**, then download `t2_t3_results.zip` from the Output tab.

In [ ]:
# ---- Settings -----------------------------------------------------------------------------
REPO_URL = "https://github.com/nvhungus/Lightweight_Context-Cluster.git"
REPO_BRANCH = "resubmission"   # the branch you pushed the T1 changes to
RUN_SMOKE_TRAINING = True      # section 2-3 (~30-40 min)
RUN_DUAL_GPU_TEST = True       # section 4 (~10 min, needs GPU T4 x2)
RUN_M1_BENCHMARK = True        # section 5 (~20-30 min)

In [ ]:
import json, os, subprocess, sys, time, zipfile
from pathlib import Path

WORK = Path("/kaggle/working")
ROOT = WORK / "Lightweight-Context-Cluster"
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fvcore", "pytest"], check=True)
print("commit:", subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

import torch
N_GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", N_GPUS, [torch.cuda.get_device_name(i) for i in range(N_GPUS)])
assert N_GPUS >= 1 and "T4" in torch.cuda.get_device_name(0), "Select the GPU T4 x2 accelerator."

RESULTS = {}  # section -> "PASS" / "FAIL: reason"


def run(cmd, env=None, log_path=None):
    """Run a command, stream its output, return the exit code."""
    print("\n$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), cwd=ROOT, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    log = open(log_path, "w") if log_path else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if log:
            log.write(line)
    if log:
        log.close()
    return proc.wait()


def find_tin_root() -> Path:
    # Bounded-depth search: never walks the 100k image files.
    for depth in range(1, 7):
        pattern = "/".join(["*"] * depth) + "/wnids.txt"
        for hit in Path("/kaggle/input").glob(pattern):
            if (hit.parent / "train").is_dir() and (hit.parent / "val" / "val_annotations.txt").exists():
                return hit.parent
    raise FileNotFoundError("Attach the Tiny-ImageNet dataset (akash2sharma/tiny-imagenet).")


TIN_ROOT = find_tin_root()
CIFAR_ROOT = WORK / "data"
print("TIN_ROOT =", TIN_ROOT)

## 1. Unit tests on Kaggle

In [ ]:
code = run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"])
RESULTS["1 pytest"] = "PASS" if code == 0 else f"FAIL: pytest exit {code}"
print(RESULTS["1 pytest"])

## 2. Smoke training of every model in the run plan (real data, T4)

Tiny-ImageNet: all baselines, HBCC-S/M, all-cluster, the 8 ablations, and 4 KD students.
CIFAR-10: ResNet-18 (teacher), HBCC-S CE, HBCC-S KD. Teachers are listed before their students.

In [ ]:
SMOKE_IDS = ["A2", "A1", "B1", "B2", "A3", "A4", "A5",
             "B3", "B4", "B5", "B6", "B7", "B8", "B9", "B10",
             "A6", "A7", "A8", "A9",
             "B11", "B15", "B18"]
SMOKE_OUT = WORK / "runs_smoke"
if RUN_SMOKE_TRAINING:
    code = run([sys.executable, "tools/run_queue.py", "--ids", *SMOKE_IDS, "--smoke", "--keep-going",
                "--output", SMOKE_OUT, "--tin-root", TIN_ROOT, "--cifar-root", CIFAR_ROOT])
    print("run_queue exit code:", code)

In [ ]:
import numpy as np, yaml

sys.path.insert(0, str(ROOT / "tools"))
import run_queue
plan = run_queue.load_plan()

rows, failures = [], []
for run_id in SMOKE_IDS:
    r = plan["by_id"][run_id]
    d = SMOKE_OUT / r.experiment
    problems = []
    for name in ["config.yaml", "run_info.json", "metrics.jsonl", "test_metrics.json",
                 "test_predictions.npy", "test_targets.npy", "best.pth", "stdout.log"]:
        if not (d / name).exists():
            problems.append(f"missing {name}")
    if not problems:
        info = json.loads((d / "run_info.json").read_text())
        cfg = yaml.safe_load((d / "config.yaml").read_text())
        preds = np.load(d / "test_predictions.npy")
        if info["seed"] != r.seed or cfg["train"]["seed"] != r.seed:
            problems.append(f"seed {info['seed']} != {r.seed}")
        if cfg["model"]["num_classes"] != (200 if r.dataset == "tin" else 10):
            problems.append(f"num_classes {cfg['model']['num_classes']}")
        if len(preds) != 2 * 256:
            problems.append(f"{len(preds)} test predictions (expected 512)")
        if not info.get("git_commit") or info.get("git_dirty"):
            problems.append("git commit missing or dirty")
        if "T4" not in (info.get("gpu_name") or ""):
            problems.append(f"gpu {info.get('gpu_name')}")
        if r.kd_teacher:
            teacher = plan["by_id"][r.kd_teacher]
            if teacher.experiment not in str(info.get("teacher_checkpoint")):
                problems.append(f"teacher {info.get('teacher_checkpoint')}")
    rows.append((run_id, r.experiment, "ok" if not problems else "; ".join(problems)))
    if problems:
        failures.append(run_id)

for row in rows:
    print(f"{row[0]:>4}  {row[1]:40s} {row[2]}")
RESULTS["2 smoke training"] = "PASS" if not failures else f"FAIL: {failures}"
print(RESULTS["2 smoke training"])

## 3. Aggregation script on the smoke runs

In [ ]:
code = run([sys.executable, "tools/aggregate_seeds.py", SMOKE_OUT, "--include-smoke",
            "--output", WORK / "results" / "smoke_summary.csv",
            "--compare", "tin_hbcc_medium_ce:tin_allcluster_medium_ce",
            "--compare", "tin_hbcc_medium_kd:tin_resnet18_ce"])
ok = code == 0 and (WORK / "results" / "smoke_summary_comparisons.json").exists()
RESULTS["3 aggregation"] = "PASS" if ok else f"FAIL: exit {code}"
print(RESULTS["3 aggregation"])

## 4. Two training jobs at once on T4 x2?

Trains HBCC-Medium on Tiny-ImageNet for 150 batches alone on GPU 0, then two copies at once
(GPU 0 and GPU 1). Both setups use 2 data workers per job. If the per-job slowdown is below
1.6x, run two queues per session (one per GPU) in the real runs.

In [ ]:
def timed_train(tag, gpu):
    out = WORK / "dual_test" / tag
    cmd = [sys.executable, "tools/train.py", "--config", "configs/hbcc_accuracy_medium.yaml",
           "--output", out, "--no-progress", "--skip-test",
           "--limit-train-batches", "150", "--limit-val-batches", "1",
           "--override", "train.epochs=1", "--override", "data.workers=2",
           "--override", "data.name=tiny_imagenet", "--override", f"data.root={TIN_ROOT}",
           "--override", "model.num_classes=200", "--override", "data.val_size=10000",
           "--override", f"experiment.name=dual_{tag}"]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(list(map(str, cmd)), cwd=ROOT, env=env,
                            stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT), out / f"dual_{tag}" / "metrics.jsonl"


def train_seconds(path):
    rec = [json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]
    return rec[0]["train_time_s"]


if RUN_DUAL_GPU_TEST and N_GPUS >= 2:
    p, m_single = timed_train("single", 0); p.wait()
    p0, m0 = timed_train("par0", 0); p1, m1 = timed_train("par1", 1); p0.wait(); p1.wait()
    single = train_seconds(m_single)
    parallel = (train_seconds(m0) + train_seconds(m1)) / 2
    ratio = parallel / single
    print(f"single job: {single:.1f}s | each of two parallel jobs: {parallel:.1f}s | slowdown x{ratio:.2f}")
    DUAL_DECISION = "USE 2 jobs per session (one per GPU)" if ratio < 1.6 else "USE 1 job per session"
    print("DECISION:", DUAL_DECISION)
    RESULTS["4 dual GPU"] = f"PASS (slowdown x{ratio:.2f} -> {DUAL_DECISION})"
else:
    RESULTS["4 dual GPU"] = "SKIPPED (needs GPU T4 x2)"
print(RESULTS["4 dual GPU"])

## 5. M1: T4 cost measurements (nothing else running)

7 CIFAR models at 32x32, batch sizes 1/16/64/128, median of 3 repeats, eager + CUDA Graph.
Self-checks against the first-submission T4 numbers: identical MACs, batch-128 peak memory within 5%
(proves the old memory column was the batch-128 peak), eager batch-1 latency within 35%, and the
CUDA-Graph output equal to eager for every model and batch size.

In [ ]:
M1_OUT = WORK / "results" / "benchmark_t4_m1"
M1_MODELS = {
    "resnet18_cifar": "configs/baselines/resnet18_cifar.yaml",
    "mobilenet_v2_cifar": "configs/baselines/mobilenet_v2_cifar.yaml",
    "shufflenet_v2_x1_0_cifar": "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
    "coc_cifar_baseline": "configs/coc_cifar_baseline.yaml",
    "hbcc_accuracy_small": "configs/hbcc_accuracy_small.yaml",
    "hbcc_accuracy_medium": "configs/hbcc_accuracy_medium.yaml",
    "hbcc_accuracy_medium_allcluster": "configs/ablations/hbcc_accuracy_medium_allcluster.yaml",
}
# First-submission T4 numbers (results/cifar10_published_results.csv): MACs, batch-128 peak MB, eager b1 ms.
OLD = {
    "resnet18_cifar": (556_659_712, 326.4, 3.03),
    "mobilenet_v2_cifar": (25_556_736, 123.4, 4.76),
    "shufflenet_v2_x1_0_cifar": (46_128_256, 94.8, 6.26),
    "coc_cifar_baseline": (34_369_600, 71.8, 11.71),
    "hbcc_accuracy_small": (92_339_392, 363.6, 8.41),
    "hbcc_accuracy_medium": (138_806_784, 481.4, 8.09),
}
if RUN_M1_BENCHMARK:
    for name, cfg in M1_MODELS.items():
        run([sys.executable, "tools/benchmark.py", "--config", cfg, "--name", name,
             "--output", M1_OUT, "--profile"], env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))

In [ ]:
m1_fail = []
print(f"{'model':34s} {'MACs+cl':>11s} {'mem b1':>7s} {'mem b128':>9s} {'eager b1':>9s} {'graph b1':>9s}  checks")
for name in M1_MODELS:
    rec = json.loads((M1_OUT / f"{name}.json").read_text())
    checks = []
    graph_keys = [k for k in rec if k.startswith("graph_matches_eager_b")]
    if not graph_keys or not all(rec[k] for k in graph_keys) or any(k.startswith("graph_error") for k in rec):
        checks.append("graph!=eager")
    if name in OLD:
        macs, mem128, lat1 = OLD[name]
        if rec["macs"] != macs:
            checks.append(f"MACs {rec['macs']}!={macs}")
        if abs(rec["peak_memory_mb_b128"] - mem128) / mem128 > 0.05:
            checks.append(f"mem128 {rec['peak_memory_mb_b128']:.1f} vs {mem128}")
        if abs(rec["latency_ms_b1"] - lat1) / lat1 > 0.35:
            checks.append(f"eager b1 {rec['latency_ms_b1']:.2f} vs {lat1}")
    if checks:
        m1_fail.append(name)
    print(f"{name:34s} {rec['macs_incl_cluster']/1e6:10.1f}M {rec['peak_memory_mb_b1']:7.1f} "
          f"{rec['peak_memory_mb_b128']:9.1f} {rec['latency_ms_b1']:8.2f}ms {rec.get('latency_graph_ms_b1', float('nan')):8.2f}ms  "
          f"{'ok' if not checks else '; '.join(checks)}")
print("GPU:", rec["device_name"], "| CPU:", rec["cpu_name"], "| torch", rec["torch_version"])
RESULTS["5 M1 benchmark"] = "PASS" if not m1_fail else f"CHECK: {m1_fail} (see table; latency drift alone is not a failure)"
print(RESULTS["5 M1 benchmark"])

## Summary and download

In [ ]:
for section, status in RESULTS.items():
    print(f"{section:20s} {status}")

bundle = WORK / "t2_t3_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for base in [WORK / "results", SMOKE_OUT]:
        if base.exists():
            for f in base.rglob("*"):
                if f.is_file() and f.suffix not in {".pth"}:
                    zf.write(f, f.relative_to(WORK))
print("download:", bundle)